# Nup133 deposition

This example shows how to use the [python-ihm library](https://github.com/ihmwg/python-ihm) programmatically to create an mmCIF file suitable for deposition at [PDB-IHM](https://pdb-ihm.org/).

This is a real integrative modeling study that was [previously published](https://pmc.ncbi.nlm.nih.gov/articles/PMC4223481/) and demonstrates how python-ihm can be used to pull together "messy" disparate output files. The original study elucidated the structure of the Nup133 protein in the *S. cerevisiae* Nuclear Pore Complex (NPC) using comparative models, small-angle X-ray (SAXS) data, electron microscopy (EM) imagery, and chemical crosslinks.

This is done by writing a Python script that reads in previously-generated outputs (for example, PDB files with coordinates, log files from various pieces of software, and tables of values that were generated for the publication), uses the python-ihm API to create a set of Python objects that describe the entire system, and then writes those objects to an mmCIF file. This Python script can then be deposited together with the modeling outputs, for example at [GitHub](https://github.com/) or [Zenodo](https://zenodo.org/), and rerun later if necessary (for example, to correct errors or to add new data, such as a citation).

This notebook can be run in [Google Colab](https://colab.research.google.com/) by opening a notebook from GitHub and then putting `integrativemodeling/nup133` into the search box, or using this [direct link](https://colab.research.google.com/github/integrativemodeling/nup133/blob/main/outputs_foxs_ensemble_new/pdb-dev/nup133.ipynb).

We must first install the Python modules used by this example using the `pip` tool, and get the actual output files used in the original modeling study from GitHub:

In [ ]:
!pip install bio ihm
%cd /content
!git clone https://github.com/integrativemodeling/nup133.git
%cd /content/nup133/outputs_foxs_ensemble_new/pdb-dev

The first step, as in most Python scripts, is to import the modules used in the script. We will be using several parts of python-ihm together with [Biopython](https://biopython.org/) (to read PDB files) and [SciPy](https://scipy.org/) (to handle spatial transforms).

In [ ]:
import ihm.dumper
import ihm.representation
import ihm.model
import ihm.reference
import ihm.citations
import ihm.protocol
import ihm.analysis
import ihm.startmodel
import ihm.metadata
import Bio.PDB
import re
import os
import csv
import glob
import numpy as np
from scipy.spatial.transform import RigidTransform, Rotation

The first Python object that we create is the top-level python-ihm [System](https://python-ihm.readthedocs.io/en/latest/main.html#ihm.System) object. All other objects will be referenced from this object:

In [ ]:
title = ("Integrative structure-function mapping of the nucleoporin "
         "Nup133 suggests a conserved mechanism for membrane anchoring "
         "of the nuclear pore complex.")
system = ihm.System(title=title)

We can add multiple [Citation](https://python-ihm.readthedocs.io/en/latest/main.html#ihm.Citation) objects. Typically the first citation should be for the modeling itself (and is marked as the 'primary' citation). A convenience function [Citation.from_pubmed_id](https://python-ihm.readthedocs.io/en/latest/main.html#ihm.Citation.from_pubmed_id) can also be used to populate this information by querying PubMed.

In [ ]:
system.citations.append(ihm.Citation(
    pmid='25139911', title=title,
    journal="Mol Cell Proteomics", volume=13, page_range=(2911, 2926),
    year=2014,
    authors=['Kim, S.J.', 'Fernandez-Martinez, J.', 'Sampathkumar, P.',
             'Martel, A.', 'Matsui, T.', 'Tsuruta, H.', 'Weiss, T.M.',
             'Shi, Y.', 'Markina-Inarrairaegui, A.', 'Bonanno, J.B.',
             'Sauder, J.M.', 'Burley, S.K.', 'Chait, B.T.', 'Almo, S.C.',
             'Rout, M.P.', 'Sali, A.'],
    doi='10.1074/mcp.M114.040915',
    is_primary=True))

We should also describe any software packages used for any part of the modeling by creating [Software](https://python-ihm.readthedocs.io/en/latest/main.html#ihm.Software) objects. These can also add a citation for the software. python-ihm provides citations for some commonly-used integrative modeling packages in its [ihm.citations](https://python-ihm.readthedocs.io/en/latest/citations.html) module:

In [ ]:
# We used HHpred to detect remote homologs for some input subunits
s = ihm.Software(
    name='HHpred', classification='protein homology detection',
    description='Protein homology detection by HMM-HMM comparison',
    version='2.0.16',
    location='https://toolkit.tuebingen.mpg.de/hhpred',
    citation=ihm.citations.hhpred)
system.software.append(s)

# We used PSIPRED to predict secondary structure for subunits
s = ihm.Software(
    name='PSIPRED', classification='secondary structure prediction',
    description='Protein secondary structure prediction based on '
                'position-specific scoring matrices',
    version='4.0',
    location='http://bioinf.cs.ucl.ac.uk/psipred/',
    citation=ihm.citations.psipred)
system.software.append(s)

# We used DISOPRED to predict (and remove) disordered regions in
# the subunits
s = ihm.Software(
    name='DISOPRED', classification='disorder prediction',
    description='prediction of protein disorder', version=3,
    location='http://bioinf.cs.ucl.ac.uk/psipred/?disopred=1',
    citation=ihm.citations.disopred)
system.software.append(s)

# We used various tools from IMP (e.g. FoXS)
s = ihm.Software(
    name="Integrative Modeling Platform (IMP)",
    version="2.2",
    classification="integrative model building",
    description="integrative model building",
    location='https://integrativemodeling.org',
    citation=ihm.citations.imp)
system.software.append(s)

# We used AllosMod for sampling
citation = ihm.Citation(
    pmid='22403063',
    title='Structure-based model of allostery predicts coupling '
          'between distant sites.', journal='Proc Natl Acad Sci U S A',
    volume=109, page_range=(4875, 4880), year=2012,
    authors=['Weinkam, P.', 'Pons, J.', 'Sali, A.'],
    doi='10.1073/pnas.1116274109')
allosmod = ihm.Software(
    name='AllosMod', classification='sampling',
    description='modeling on a custom energy landscape',
    location='https://salilab.org/allosmod',
    citation=citation)

Now we can move on to describe the chemical structure itself. This starts with the primary sequence of any polymers, as available in a sequence database. In the case of Nup133 we have just a single polypeptide chain and its sequence is available in UniProt:

In [ ]:
# Sequence in UniProt
ref = ihm.reference.UniProtSequence.from_accession('P36161')

If the sequence differs between the reference database and our model we must add alignment information and/or a description of any mutations, insertions or deletions. See the [ihm.reference](https://python-ihm.readthedocs.io/en/latest/reference.html) module for more information. For Nup133 there is only a small difference at the C terminus:

In [ ]:
ref.alignments.append(ihm.reference.Alignment(db_begin=2, entity_begin=3))

To describe what was actually modeled, we need to create one or more [Entity](https://python-ihm.readthedocs.io/en/latest/main.html#ihm.Entity) objects. An Entity needs the primary sequence, which we extract from one of our output PDB files using Biopython. We also provide it with the reference database information:

In [ ]:
def get_sequence(pdb_file):
    """Get the primary sequence of a given PDB file"""
    p = Bio.PDB.PDBParser()
    s = p.get_structure('rep', pdb_file)
    ppb = Bio.PDB.PPBuilder()
    pp, = ppb.build_peptides(s)
    return str(pp.get_sequence())

# System is a single chain - extract its sequence from one of the
# output PDBs (all models should have the same sequence)
seq = get_sequence('../MES_results/Int325_6_B10770001_MSE.pdb')
entity = ihm.Entity(seq, description='Nup133',
                    references=[ref])
system.entities.append(entity)

An Entity describes a unique sequence. Each *instance* of that sequence corresponds to an [AsymUnit](https://python-ihm.readthedocs.io/en/latest/main.html#ihm.AsymUnit). (For example, a homodimer protein would contain a single Entity but two AsymUnits, while a heterodimer would contain two Entities and two AsymUnits.) The PDB-IHM database always numbers polymers sequentially starting from 1 (their `seq_id`). If we want to use a different "author-provided" numbering, we can provide that using the `auth_seq_id_map` argument:

In [ ]:
# Note that our published model is numbered from 0, i.e. offset by -1
# from the internal 1-based mmCIF numbering
asym = ihm.AsymUnit(entity, details='Nup133', auth_seq_id_map=-1)
system.asym_units.append(asym)

We often need to describe parts of the system, such as individual chains or parts of them. This is handled by the [Assembly](https://python-ihm.readthedocs.io/en/latest/main.html#ihm.Assembly) class. Here we create an assembly that spans the whole system:

In [ ]:
assembly = ihm.Assembly([asym], name='Modeled assembly')

Now we can describe how each part of the system was represented in the modeling. This may include the initial guess or "starting model" used, and/or whether the model consisted of atoms or a coarse-grained representation. python-ihm provides a [StartingModel](https://python-ihm.readthedocs.io/en/latest/startmodel.html#ihm.startmodel.StartingModel) class to provide the information about the starting model, but we must provide an implementation for its `get_atoms` method to actually read the coordinates. Here we use Biopython to read the starting model from a PDB file and return ihm Atom objects:

In [ ]:
class StartingModel(ihm.startmodel.StartingModel):
    # Override StartingModel so we can provide coordinates

    def __init__(self, file_name, **kwargs):
        super().__init__(**kwargs)
        self.file_name = file_name

    def get_atoms(self):
        # Read atoms from our starting model using Biopython, and yield
        # ihm.model.Atom objects
        p = Bio.PDB.PDBParser()
        s = p.get_structure('rep', self.file_name)
        for model in s:
            for nchain, chain in enumerate(model):
                assert nchain == 0   # should only be one chain
                for nres, residue in enumerate(chain):
                    for atom in residue:
                        coord = atom.get_vector()
                        yield ihm.model.Atom(
                            asym_unit=self.asym_unit, seq_id=nres+1,
                            atom_id=atom.get_id(), x=coord[0], y=coord[1],
                            z=coord[2], type_symbol=atom.element,
                            biso=atom.get_bfactor())

We can then write a simple function that reads a single chain from one of our input PDB files (in this case, they are [Modeller](https://salilab.org/modeller/) models). We also use python-ihm's [ihm.metadata](https://python-ihm.readthedocs.io/en/latest/metadata.html) module here that reads additional metadata from the PDB file. This includes information like the PDB ID, the software used to generate it, any comparative modeling templates, and so on:

In [ ]:
def get_starting_modeller_model(asym, pdb_file, offset, chain):
    """Read the MODELLER model that we used as a starting guess for
       our modeling, and return it as an ihm.startmodel object."""
    p = ihm.metadata.PDBParser()
    m = p.parse_file(pdb_file)
    return StartingModel(file_name=pdb_file, asym_unit=asym,
                         dataset=m['dataset'], asym_id=chain,
                         templates=m['templates'][chain], offset=offset,
                         software=m['software'][0],
                         script_file=m['script'])

Now we can use this function to read in the starting Modeller model and describe the representation of the system. For Nup133 we represent all of the single AsymUnit atomically:

In [ ]:
# Atomic model, starting from a Modeller comparative model
modeller_model = get_starting_modeller_model(
    asym, pdb_file='../../MODELLER/combined/23904.B99990030.pdb',
    offset=1, chain='A')
rep = ihm.representation.Representation(
        [ihm.representation.AtomicSegment(asym, rigid=False,
                                          starting_model=modeller_model)])

The next step is to add information on the experimental data used in the modeling. Generally this consists of
 - the actual input dataset (see the [ihm.dataset](https://python-ihm.readthedocs.io/en/latest/dataset.html) module for more information) that generally points to a domain-specific database such as [EMDB](https://www.ebi.ac.uk/emdb/), [BMRB](https://bmrb.io/) or [PRIDE](https://www.ebi.ac.uk/pride/) or a file deposited with a DOI at a service such as [Zenodo](https://zenodo.org/) (see [ihm.location](https://python-ihm.readthedocs.io/en/latest/location.html) for more information).
 - a restraint object (see the [ihm.restraint](https://python-ihm.readthedocs.io/en/latest/restraint.html) module for more information) that describes how the data was used in the modeling, the subset of the model (as an Assembly object) that the data applied to, and so on.
 - fit objects that describe how well individual models fit the data.

First we add the SAXS data by parsing summary CSV files and other files output by the [FoXS](https://modbase.compbio.ucsf.edu/foxs/) tool: 

In [ ]:
class SAXSFits:
    """Parse the SAXS csv file and add suitable fit data to the mmCIF file"""
    seqrange_re = re.compile(r'(\d+)\s*\-\s*(\d+)')

    def __init__(self, asym, saxs_dir):
        self.asym, self.saxs_dir = asym, saxs_dir

    def add_from_csv(self):
        with open(os.path.join(self.saxs_dir, 'Table6_SAXS.csv'), newline='') as fh:
            for row in csv.DictReader(fh):
                if row['FoXS fit score'] and row['Protein'] == 'NUP133':
                    yield self._add_one(row)

    def _add_one(self, row):
        protid = row['Protein ID']
        profile = (glob.glob('%s/%s_*.sub' % (self.saxs_dir, protid))
                   + glob.glob('%s/%s_*.dat' % (self.saxs_dir, protid)))[0]
        loc = ihm.location.InputFileLocation(
            profile, details=row['Notes'] if row['Notes'] else None)
        dataset = ihm.dataset.SASDataset(location=loc)
        m = self.seqrange_re.match(row['Sequence coverage'])
        seqrange = (int(m.group(1)), int(m.group(2)))
        assembly = ihm.Assembly([self.asym(*seqrange)], name='SAXS assembly')
        r = ihm.restraint.SASRestraint(
            dataset=dataset, assembly=assembly,
            segment=False, fitting_method='FoXS', multi_state=True,
            radius_of_gyration=row['Rg'])
        r._fit_score_all_models = row['FoXS fit score']
        return r

    def add_model(self, model, restraints):
        """Add fit data for the given model against all restraints"""
        for r in restraints:
            r.fits[model] = ihm.restraint.SASRestraintFit(
                                    chi_value=r._fit_score_all_models)

saxs_fits = SAXSFits(asym, saxs_dir='../../SAXS')
saxs_restraints = list(saxs_fits.add_from_csv())
system.restraints.extend(saxs_restraints)

Next we do something similar for the electron microscopy class average data used in the modeling:

In [ ]:
def get_centroid(fname):
    """Return the coordinates of the centroid of the given PDB file"""
    p = Bio.PDB.PDBParser()
    s = p.get_structure('rep', fname)
    atoms = [a for model in s for chain in model for residue in chain
             for a in residue if a.name == 'CA']
    coords = np.array([a.get_vector().get_array() for a in atoms])
    return np.mean(coords, axis=0)
    
def get_registrations(fname):
    """Yield information from the given registration parameters file"""
    with open(fname) as fh:
        for line in fh:
            if line.startswith('#') or '|' not in line:
                continue
            spl = line.split('|')
            # Image #, quaternion plus x and y shift
            yield (int(spl[0]), [float(spl[i]) for i in range(5, 9)],
                   [float(spl[i]) for i in range(9, 11)])

def get_transformations(registrations, centroid, pixel_size, image_size):
    """Yield a set of transformations from the given registration file"""
    # See do_project_particles() in IMP's modules/em2d/src/project.cpp for
    # the underlying algorithm used here.

    # Reported rotation quaternion assumes both the model centroid and the
    # center of the class average are at the origin, so we need to translate
    # accordingly to get the transformation that puts the model onto the
    # class average
    to_centroid = RigidTransform.from_translation(-centroid)
    image_origin = pixel_size * image_size * 0.5

    cleanup = RigidTransform.from_translation([image_origin, image_origin, 0.])

    for num, quat, shift in get_registrations(registrations):
        rot = Rotation.from_quat(quat, scalar_first=True)
        # Reported shift is in pixels, so convert to angstroms
        tnsl = [pixel_size * shift[0], pixel_size * shift[1], 0.]
        trans = RigidTransform.from_components(tnsl, rot)
        trans = cleanup * trans * to_centroid
        yield num, trans


class EM2DFits:

    num_images = 23
    image_name_format = '23904_%02d.pgm'

    def __init__(self, assembly, em2d_dir, csv_file):
        self.assembly, self.em2d_dir = assembly, em2d_dir
        self.im_per_class, self.ccc = self.read_csv(csv_file)

    def read_csv(self, fname):
        """Parse Table S3 to get # of images and CCC per class"""
        im_per_class = [None] * self.num_images
        ccc = [None] * self.num_images
        with open(fname, newline='') as fh:
            for row in csv.reader(fh):
                try:
                    class_id = int(row[0]) - 1
                    im_per_class[class_id], ccc[class_id] = \
                        self._parse_csv_fit(row)
                except ValueError:
                    continue
        return im_per_class, ccc

    def _parse_csv_fit(self, row):
        ccc = [None]*4
        z = [None]*4
        (class_id, im_per_class, cutoff, z[0], ccc[0], z[1], ccc[1],
         z[2], ccc[2], z[3], ccc[3]) = row
        im_per_class = int(im_per_class)
        z = [float(x) for x in z]
        ccc = [float(x) for x in ccc]
        return im_per_class, ccc

    def add_all(self):
        """Point to the raw image files used for the EM2D restraint"""
        for image in range(self.num_images):
            fname = os.path.join(self.em2d_dir, self.image_name_format % image)
            loc = ihm.location.InputFileLocation(fname)
            dataset = ihm.dataset.EM2DClassDataset(location=loc)
            r = ihm.restraint.EM2DRestraint(
                dataset=dataset, assembly=self.assembly, segment=False,
                # From ../2_em2d_0.sh
                pixel_size_width=2.28739, pixel_size_height=2.28739,
                number_raw_micrographs=self.im_per_class[image],
                number_of_projections=1000, image_resolution=15.)
            yield r

    def add_model(self, nmodel, model, restraints):
        """Add fit data to the given model for all restraints (images)"""
        assert len(restraints) == self.num_images

        centroid = get_centroid(model.file_name)
        pixel_size = 2.28739
        image_size = 128  # todo: check
        regparam = os.path.join(model.file_name[:-4], 'registration.params')
        # Convert registration parameters to standard transformation matrices
        for num, trans in get_transformations(regparam, centroid,
                                              pixel_size, image_size):
            restraints[num].fits[model] = ihm.restraint.EM2DRestraintFit(
                rot_matrix=trans.rotation.as_matrix(),
                tr_vector=trans.translation,
                cross_correlation_coefficient=self.ccc[num][nmodel])


em2d_fits = EM2DFits(assembly, em2d_dir='../ISAC_p150_t346_m30',
                     csv_file='../MES_results/Nup133_tableS3.csv')
em2d_restraints = list(em2d_fits.add_all())
system.restraints.extend(em2d_restraints)

Finally, chemical crosslinking data is added in a similar fashion:

In [ ]:
class CrossLinkFits:

    cross_linkers = {'DSS': ihm.cross_linkers.dss,
                     'EDC': ihm.cross_linkers.edc}

    def __init__(self, asym, xlink_dir):
        self.asym, self.xlink_dir = asym, xlink_dir

    def add_restraints(self):
        """Parse the crosslinks text file and return a set of restraints"""
        fname = os.path.join(self.xlink_dir, 'DSS_EDC_crosslinks.txt')
        loc = ihm.location.InputFileLocation(fname)
        d = ihm.dataset.CXMSDataset(loc)
        for typ in ('DSS', 'EDC'):
            with open(fname) as fh:
                yield self.get_intra_restraint(d, typ, fh, self.asym)

    def get_intra_restraint(self, dataset, linker_type, fh, asym):
        # Distance thresholds, from Table II
        threshold = {'DSS': 35., 'EDC': 25.}
        distance = ihm.restraint.UpperBoundDistanceRestraint(
            threshold[linker_type])
        r = ihm.restraint.CrossLinkRestraint(
            dataset, linker=self.cross_linkers[linker_type])
        for line in fh:
            if line.startswith(linker_type):
                break
        for line in fh:
            if line == '\n':
                continue
            resids = line.split(',')
            if len(resids) == 2:
                # Numbering of xlink file is off by one compared
                # to output model
                res = [asym.entity.residue(int(x) + 1) for x in resids]
                xxl = ihm.restraint.ExperimentalCrossLink(res[0], res[1])
                r.experimental_cross_links.append([xxl])
                xl = ihm.restraint.ResidueCrossLink(xxl, asym, asym, distance)
                r.cross_links.append(xl)
            else:
                return r


xlink_fits = CrossLinkFits(asym, xlink_dir='../../Crosslinks')
xlink_restraints = list(xlink_fits.add_restraints())
system.restraints.extend(xlink_restraints)

We can also group datasets together. Here we create two such groups - one for all of the SAXS and EM data, and another for all of the crosslinks:

In [ ]:
saxs_em2d_datasets = ihm.dataset.DatasetGroup(
    r.dataset for r in saxs_restraints + em2d_restraints)
xlink_datasets = ihm.dataset.DatasetGroup(r.dataset for r in xlink_restraints)

Now we describe how the modeling was done using classes from the [ihm.protocol](https://python-ihm.readthedocs.io/en/latest/protocol.html) module. A modeling protocol can consist of multiple steps using different methods and combinations of input data:

In [ ]:
# Describe the modeling protocol
protocol = ihm.protocol.Protocol(name='Modeling')
# Taking the original comparative model as input, first we ran AllosMod,
# which didn't use the experimental datasets:
protocol.steps.append(ihm.protocol.Step(
    assembly=assembly, dataset_group=None,
    method='AllosMod',
    name='MD-based conformational sampling',
    num_models_begin=1, num_models_end=7000,
    software=allosmod))
# Next we ran MES using both saxs and em2d:
protocol.steps.append(ihm.protocol.Step(
    assembly=assembly, dataset_group=saxs_em2d_datasets,
    method='MES',
    name='Minimal Ensemble Search',
    num_models_begin=7000, num_models_end=4, multi_state=True))
# Finally we validated against crosslinks
analysis = ihm.analysis.Analysis()
analysis.steps.append(ihm.analysis.ValidationStep(
    assembly=assembly, dataset_group=xlink_datasets,
    feature='other', num_models_begin=4, num_models_end=4))
protocol.analyses.append(analysis)

Now we can add the coordinates of the structural models themselves. This is done using the [Model](https://python-ihm.readthedocs.io/en/latest/model.html#ihm.model.Model) class. Just as for StartingModel above, we must provide a set of atoms (or spheres, for coarse-grained models), and again we do this by reading coordinates from a PDB file using Biopython:

In [ ]:
class PDBModel(ihm.model.Model):
    """Pass a Biopython model through to IHM"""
    def __init__(self, file_name, asym_units, **kwargs):
        super().__init__(**kwargs)
        self.file_name = file_name
        self.asym_units = asym_units

    def get_atoms(self):
        # Use Biopython to read the structure from a PDB file, and then yield
        # a set of ihm.model.Atom objects
        p = Bio.PDB.PDBParser()
        s = p.get_structure('rep', self.file_name)
        for model in s:
            for nchain, chain in enumerate(model):
                asym = self.asym_units[nchain]
                for nres, residue in enumerate(chain):
                    for atom in residue:
                        coord = atom.get_vector()
                        yield ihm.model.Atom(
                            asym_unit=asym, seq_id=nres+1,
                            atom_id=atom.get_id(), x=coord[0], y=coord[1],
                            z=coord[2], type_symbol=atom.element)

PDB-IHM is quite flexible in terms of the actual structural models that can be deposited. Multiple models can be provided, they may differ in composition, may span multiple scales and multiple states, and may be part of an ensemble or an ordered process. See the classes in the [ihm.model](https://python-ihm.readthedocs.io/en/latest/model.html) module for more information. For Nup133 we deposited multiple states that were found using a minimal ensemble search (MES) method. Here we parse the MES log file to determine the population fractions for each state, then use the `PDBModel` class (above) to add the coordinates of the model to our python-ihm system. Finally we can add information on the fit of each model to the SAXS and EM data:

In [ ]:
def get_models_with_fractions(mes_dir):
    """Parse the MES log file to get the names of the output models and their
       population fractions"""
    def skip_to_models(log):
        for line in log:
            if line.startswith('    filename weight binary_score bit_count'):
                return
        raise ValueError("Bad MES log file")
    model_num = -1
    with open(os.path.join(mes_dir, 'new_mes4.log')) as log:
        skip_to_models(log)
        for model_line in log:
            if model_line == '\n':
                return
            s = model_line.strip().split(' ')
            model_num += 1
            yield (model_num, os.path.join(mes_dir, s[0][:-4] + '.pdb'),
                   float(s[1]))

# Output the final models selected by MES. Each is represented as a state
# with given population fraction
g = ihm.model.StateGroup()
for nmodel, model, fraction in get_models_with_fractions(
        mes_dir='../MES_results'):
    m = PDBModel(assembly=assembly, protocol=protocol, representation=rep,
                 file_name=model, asym_units=[asym])
    saxs_fits.add_model(m, saxs_restraints)
    em2d_fits.add_model(nmodel, m, em2d_restraints)
    s = ihm.model.State([ihm.model.ModelGroup([m])],
                        type='minimal ensemble',
                        name='Minimal ensemble conformation',
                        experiment_type='Fraction of bulk',
                        population_fraction=fraction)
    g.append(s)
system.state_groups.append(g)

If we referenced any other file by filename (rather than by a database identifier such as a PDB ID) then the final mmCIF file will reference that filename (rather than incorporating the contents of the file). These files must be deposited in a public location and assigned a DOI so that users of the mmCIF file can access them. Assuming that all files in the GitHub repository were deposited at Zenodo, the [update_locations_in_repositories](https://python-ihm.readthedocs.io/en/latest/main.html#ihm.System.update_locations_in_repositories) function can be used to replace local paths with pointers to these deposited files:

In [ ]:
# Replace local paths with pointers to deposited files at Zenodo
# (Note that the real Nup133 deposition split the GitHub repo into
# multiple Zenodo deposits.)
repos = []
repos.append(ihm.location.Repository(
             doi="10.5281/zenodo.1209565", root="../..",
             url="https://zenodo.org/record/1209565/files/nup133-master.zip",
             top_directory="nup133-master"))
system.update_locations_in_repositories(repos)

Finally, the complete ihm System object can be written out to a file. python-ihm supports writing both in the text mmCIF format and in the more compact BinaryCIF format:

In [ ]:
# Write out in mmCIF and BinaryCIF format
with open('nup133.cif', 'w') as fh:
    ihm.dumper.write(fh, [system])
with open('nup133.bcif', 'wb') as fh:
    ihm.dumper.write(fh, [system], format='BCIF')

The resulting mmCIF file can be downloaded from Google Colab using the Files tab (it can be found in the `/content/nup133/outputs_foxs_ensemble_new/pdb-dev` directory) and then uploaded to PDB-IHM or viewed in a viewer such as [ChimeraX](https://www.cgl.ucsf.edu/chimerax/) or [Mol*](https://molstar.org/).